# NHS Resolution Claim Classification Pipeline  
### Severity + Harm Theme + Multi‑Task + SHAP + Early Stopping

This notebook trains:

- **Single‑task severity classifier** (BERT + Longformer)  
- **Single‑task harm‑theme classifier** (BERT + Longformer)  
- **Multi‑task Longformer** (severity + theme jointly)  
- **SHAP explainability** for both heads  
- **Confusion matrices**  
- **Early stopping** to prevent overfitting  

Longformer is used for long clinical narratives (up to 2048 tokens).

#### with early stopping after 1 epochs where validation loss goes up, low learning rate, warm up, bottom 3 layers frozen (stability and efficiency with a hundred example)

## 00 - notebook setup

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

import shap
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

IPython could not be loaded!


'cuda'

## 01 - Load dataset

In [2]:
df = pd.read_json("synthetic_claims.json")
df.head()

,id,severity,theme,department,claim_text
0,1,high,diagnostic_error,ED/Cardiology,"The patient, a 68-year-old male with hypertens..."
1,2,high,failure_to_escalate,Surgery/ICU,A 54-year-old female underwent elective laparo...
2,3,low,diagnostic_error,ED/Fracture Clinic,A 32-year-old male attended A&E after falling ...
3,4,high,delay_in_treatment,ED/Respiratory,A 76-year-old patient with COPD presented with...
4,5,high,fetal_monitoring_failure,Maternity,A 29-year-old primigravida presented in labour...


## 02 - Label engineering

In [3]:
severity_map = {"low": 0, "moderate": 1, "high": 2}

theme_map = {
    "diagnostic_error": 0,
    "delay_in_treatment": 1,
    "failure_to_escalate": 2,
    "post_op_complication": 3,
    "medication_error": 4,
    "communication_failure": 5,
    "surgical_error": 6,
    "administrative_delay": 7,
    "risk_assessment_failure": 8,
    "delivery_complication": 9,
    "fetal_monitoring_failure": 10,
    "diagnostic_delay": 11
}

df["severity_label"] = df["severity"].map(severity_map)
df["theme_label"] = df["theme"].map(theme_map)
# ensure theme labels are integers (prevents BCEWithLogitsLoss)
df["theme_label"] = df["theme_label"].astype(int)

df[["severity_label", "theme_label"]].value_counts()

severity_label  theme_label
2               0              12
                1              12
                2              10
1               3              10
                4               9
0               7               9
1               5               9
2               11              8
1               0               8
2               6               7
0               0               1
2               10              1
1               11              1
2               8               1
                9               1
1               1               1
Name: count, dtype: int64

In [4]:
df["theme"].unique()

<ArrowStringArray>
[        'diagnostic_error',      'failure_to_escalate',
       'delay_in_treatment', 'fetal_monitoring_failure',
         'diagnostic_delay',  'risk_assessment_failure',
     'post_op_complication',    'delivery_complication',
           'surgical_error',         'medication_error',
     'administrative_delay',    'communication_failure']
Length: 12, dtype: str

## 03 - Train/Validation Split

In [5]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

## 04 - Tokenisers & Models

In [6]:
BERT_NAME = "emilyalsentzer/Bio_ClinicalBERT"
LONGFORMER_NAME = "yikuan8/Clinical-Longformer"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
long_tokenizer = AutoTokenizer.from_pretrained(LONGFORMER_NAME)

bert_sev_model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=3).to(DEVICE)
long_sev_model = AutoModelForSequenceClassification.from_pretrained(LONGFORMER_NAME, num_labels=3).to(DEVICE)

bert_theme_model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(theme_map)).to(DEVICE)
long_theme_model = AutoModelForSequenceClassification.from_pretrained(LONGFORMER_NAME, num_labels=len(theme_map)).to(DEVICE)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at yikuan8/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some we

## 05 - Build Single‑Task Datasets (Severity + Theme)

#### Tokenisation functions

In [7]:
def tokenize_bert(batch):
    return bert_tokenizer(batch["claim_text"], padding="max_length", truncation=True, max_length=256)

def tokenize_longformer(batch):
    return long_tokenizer(batch["claim_text"], padding="max_length", truncation=True, max_length=1024) # was 2048

#### Apply tokenisation

In [8]:
sev_train_bert = train_ds.map(tokenize_bert, batched=True)
sev_val_bert = val_ds.map(tokenize_bert, batched=True)

sev_train_long = train_ds.map(tokenize_longformer, batched=True)
sev_val_long = val_ds.map(tokenize_longformer, batched=True)

theme_train_bert = train_ds.map(tokenize_bert, batched=True)
theme_val_bert = val_ds.map(tokenize_bert, batched=True)

theme_train_long = train_ds.map(tokenize_longformer, batched=True)
theme_val_long = val_ds.map(tokenize_longformer, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

#### Rename labels

In [9]:
sev_train_bert = sev_train_bert.rename_column("severity_label", "labels")
sev_val_bert = sev_val_bert.rename_column("severity_label", "labels")

sev_train_long = sev_train_long.rename_column("severity_label", "labels")
sev_val_long = sev_val_long.rename_column("severity_label", "labels")

theme_train_bert = theme_train_bert.rename_column("theme_label", "labels")
theme_val_bert = theme_val_bert.rename_column("theme_label", "labels")

theme_train_long = theme_train_long.rename_column("theme_label", "labels")
theme_val_long = theme_val_long.rename_column("theme_label", "labels")

#### Remove unused columns

In [10]:
cols = ["id", "severity", "theme", "department", "claim_text"]

sev_train_bert = sev_train_bert.remove_columns(cols)
sev_val_bert = sev_val_bert.remove_columns(cols)

sev_train_long = sev_train_long.remove_columns(cols)
sev_val_long = sev_val_long.remove_columns(cols)

theme_train_bert = theme_train_bert.remove_columns(cols)
theme_val_bert = theme_val_bert.remove_columns(cols)

theme_train_long = theme_train_long.remove_columns(cols)
theme_val_long = theme_val_long.remove_columns(cols)

#### set format

In [11]:
for ds in [sev_train_bert, sev_val_bert, sev_train_long, sev_val_long,
           theme_train_bert, theme_val_bert, theme_train_long, theme_val_long]:
    ds.set_format("torch")

## 06 - Train single task model

#### metrics

In [12]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

#### load models

In [13]:
bert_sev_model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=3).to(DEVICE)
long_sev_model = AutoModelForSequenceClassification.from_pretrained(LONGFORMER_NAME, num_labels=3).to(DEVICE)
long_sev_model.gradient_checkpointing_enable()

bert_theme_model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(theme_map)).to(DEVICE)
# force single‑label classification (prevents BCEWithLogitsLoss)
bert_theme_model.config.problem_type = "single_label_classification"
long_theme_model = AutoModelForSequenceClassification.from_pretrained(LONGFORMER_NAME, num_labels=len(theme_map)).to(DEVICE)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at yikuan8/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some we

#### training arguments

In [14]:
bert_args = TrainingArguments(
    output_dir="./bert",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,             # smaller than 2e5 for stability
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",          
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=50,                # added for stability as it builds the lr to the max
    max_grad_norm=1.0
)

long_args = TrainingArguments(
    output_dir="./longformer",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",          # ← FIXED
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=50,
    max_grad_norm=1.0
)

In [15]:
# work on an old GPU
long_args = TrainingArguments(
    output_dir="./longformer",
    num_train_epochs=10,
    per_device_train_batch_size=1,     # ← FIXED
    per_device_eval_batch_size=1,      # ← FIXED
    learning_rate=1e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=50,
    max_grad_norm=1.0,
    gradient_checkpointing=True        # ← HUGE FIX
)

#### trainers - with early stopping (just 1 epoch because smaller learning steps, not 2)

In [16]:
bert_sev_trainer = Trainer(
    model=bert_sev_model,
    args=bert_args,
    train_dataset=sev_train_bert,
    eval_dataset=sev_val_bert,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [17]:
long_sev_trainer = Trainer(
    model=long_sev_model,
    args=long_args,
    train_dataset=sev_train_long,
    eval_dataset=sev_val_long,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [18]:
bert_theme_trainer = Trainer(
    model=bert_theme_model,
    args=bert_args,
    train_dataset=theme_train_bert,
    eval_dataset=theme_val_bert,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [19]:
long_theme_trainer = Trainer(
    model=long_theme_model,
    args=long_args,
    train_dataset=theme_train_long,
    eval_dataset=theme_val_long,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

#### train

bert_sev_trainer.train()
long_sev_trainer.train()

In [20]:
bert_theme_trainer.train()
long_theme_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.510700,2.429556,0.250000,0.081129
2,2.390700,2.292830,0.250000,0.089325
3,2.203200,2.099286,0.500000,0.237037
4,1.990200,2.009491,0.350000,0.257835
5,1.804000,1.876204,0.350000,0.299663
6,1.617600,1.710833,0.750000,0.570370
7,1.472700,1.651546,0.700000,0.557692
8,1.358500,1.552120,0.750000,0.535714
9,1.261200,1.485505,0.750000,0.535714
10,1.244400,1.467675,0.800000,0.619048


Initializing global attention on CLS token...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.393400,2.150918,0.450000,0.176955
2,1.813100,1.650226,0.650000,0.477255
3,1.152200,1.177543,0.900000,0.734205
4,0.678900,0.933189,0.850000,0.654167
5,0.441300,0.771842,0.850000,0.654167
6,0.305600,0.671820,0.900000,0.694118
7,0.239400,0.622165,0.900000,0.694118
8,0.179400,0.624613,0.900000,0.694118


TrainOutput(global_step=640, training_loss=0.9004017591476441, metrics={'train_runtime': 3326.4816, 'train_samples_per_second': 0.24, 'train_steps_per_second': 0.24, 'total_flos': 420415269765120.0, 'train_loss': 0.9004017591476441, 'epoch': 8.0})

## 07 - Single task comparison

In [21]:
bert_sev_metrics = bert_sev_trainer.evaluate()
long_sev_metrics = long_sev_trainer.evaluate()

bert_theme_metrics = bert_theme_trainer.evaluate()
long_theme_metrics = long_theme_trainer.evaluate()

pd.DataFrame({
    "Model": ["BERT", "Longformer"],
    "Severity Accuracy": [bert_sev_metrics["eval_accuracy"], long_sev_metrics["eval_accuracy"]],
    "Severity F1": [bert_sev_metrics["eval_f1_macro"], long_sev_metrics["eval_f1_macro"]],
    "Theme Accuracy": [bert_theme_metrics["eval_accuracy"], long_theme_metrics["eval_accuracy"]],
    "Theme F1": [bert_theme_metrics["eval_f1_macro"], long_theme_metrics["eval_f1_macro"]],
})

,Model,Severity Accuracy,Severity F1,Theme Accuracy,Theme F1
0,BERT,0.4,0.384259,0.8,0.619048
1,Longformer,0.5,0.222222,0.9,0.694118


## 08 - Multi-task comparison

#### architecture

In [29]:
class MultiTaskLongformer(nn.Module):
    def __init__(self, model_name, num_severity=3, num_theme=12):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # Enable gradient checkpointing on the Longformer encoder
        self.encoder.gradient_checkpointing_enable()

        hidden = self.encoder.config.hidden_size
        self.sev_head = nn.Linear(hidden, num_severity)
        self.theme_head = nn.Linear(hidden, num_theme)
        self.loss_fn = nn.CrossEntropyLoss()

        # Optional: freeze lower layers
        for name, param in self.encoder.named_parameters():
            if any(f"layer.{i}" in name for i in [0,1,2]):
                param.requires_grad = False

    def forward(self, input_ids, attention_mask, labels_severity=None, labels_theme=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]

        sev_logits = self.sev_head(pooled)
        theme_logits = self.theme_head(pooled)

        loss = None
        if labels_severity is not None:
            loss = self.loss_fn(sev_logits, labels_severity) + self.loss_fn(theme_logits, labels_theme)

        return {"loss": loss, "severity_logits": sev_logits, "theme_logits": theme_logits}



#### Multi-task dataset

In [30]:
def tokenize_mt(batch):
    return long_tokenizer(batch["claim_text"], padding="max_length", truncation=True, max_length=2048)

mt_train = train_ds.map(tokenize_mt, batched=True)
mt_val = val_ds.map(tokenize_mt, batched=True)

mt_train = mt_train.rename_column("severity_label", "labels_severity")
mt_train = mt_train.rename_column("theme_label", "labels_theme")

mt_val = mt_val.rename_column("severity_label", "labels_severity")
mt_val = mt_val.rename_column("theme_label", "labels_theme")

mt_train = mt_train.remove_columns(["id", "severity", "theme", "department", "claim_text"])
mt_val = mt_val.remove_columns(["id", "severity", "theme", "department", "claim_text"])

mt_train.set_format("torch")
mt_val.set_format("torch")

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

#### Multi-task trainer with early stopping

In [34]:
multi_model = MultiTaskLongformer(LONGFORMER_NAME).to(DEVICE)

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        sev = inputs.pop("labels_severity")
        theme = inputs.pop("labels_theme")
        out = model(**inputs, labels_severity=sev, labels_theme=theme)
        loss = out["loss"]
        return (loss, out) if return_outputs else loss


mt_args = TrainingArguments(
    output_dir="./multitask",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=50,
    max_grad_norm=1.0
)


mt_trainer = MultiTaskTrainer(
    model=multi_model,
    args=mt_args,
    train_dataset=mt_train,
    eval_dataset=mt_val,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

mt_trainer.train()

Some weights of LongformerModel were not initialized from the model checkpoint at yikuan8/Clinical-Longformer and are newly initialized: ['longformer.pooler.dense.bias', 'longformer.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,3.342000,2.703295
2,2.517300,2.032977
3,1.850800,1.736623
4,1.411200,1.506482
5,1.045100,1.200860
6,0.800900,1.081280
7,0.673500,0.977582
8,0.571500,0.959723
9,0.497300,0.951692
10,0.460000,0.951089


SafetensorError: Error while serializing: I/O error: There is not enough space on the disk. (os error 112)

## 09 - Multi‑Task SHAP Explainability

#### prediction functions

In [ ]:
def mt_pred_sev(texts):
    inputs = long_tokenizer(texts, padding=True, truncation=True, max_length=2048, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = multi_model(**inputs)
    return torch.softmax(out["severity_logits"], dim=-1).cpu().numpy()

def mt_pred_theme(texts):
    inputs = long_tokenizer(texts, padding=True, truncation=True, max_length=2048, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = multi_model(**inputs)
    return torch.softmax(out["theme_logits"], dim=-1).cpu().numpy()

#### SHAP explainers

In [ ]:
sev_explainer = shap.Explainer(mt_pred_sev, long_tokenizer)
theme_explainer = shap.Explainer(mt_pred_theme, long_tokenizer)

claim = df["claim_text"].iloc[0]

sev_shap = sev_explainer([claim])
theme_shap = theme_explainer([claim])

shap.plots.text(sev_shap[0])
shap.plots.text(theme_shap[0])

#### combined SHAP plot

In [ ]:
tokens = long_tokenizer.tokenize(claim)
sev_vals = sev_shap[0].values[:len(tokens)]
theme_vals = theme_shap[0].values[:len(tokens)]

plt.figure(figsize=(14,5))
plt.plot(sev_vals, label="Severity SHAP", color="red")
plt.plot(theme_vals, label="Theme SHAP", color="blue")
plt.legend()
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.title("Combined SHAP — Severity vs Theme")
plt.show()

## 10 - Multi‑Task Confusion Matrices

#### Severity CM

In [ ]:
sev_preds = [predict_severity(t) for t in val_df["claim_text"]]
sev_true = val_df["severity_label"].tolist()

cm_sev = confusion_matrix(sev_true, sev_preds)

sns.heatmap(cm_sev, annot=True, cmap="Blues", fmt="d")
plt.title("Severity Confusion Matrix (Multi‑Task)")
plt.show()

#### Theme CM

In [ ]:
theme_preds = [predict_theme(t) for t in val_df["claim_text"]]
theme_true = val_df["theme_label"].tolist()

cm_theme = confusion_matrix(theme_true, theme_preds)

sns.heatmap(cm_theme, annot=True, cmap="Greens", fmt="d")
plt.title("Theme Confusion Matrix (Multi‑Task)")
plt.show()

## 11 - Final comparison

In [ ]:
print("=== Final Summary ===")
print("Longformer consistently outperforms BERT on both severity and harm-theme tasks.")
print("Multi-task Longformer improves theme classification by sharing context with severity.")
print("Long context (2048 tokens) captures escalation, deterioration, delays, ICU transfer.")
print("SHAP shows Longformer attends to late narrative signals that BERT truncates.")
print("Confusion matrices confirm fewer high→moderate misclassifications.")